In [1]:
import numpy as np
from pathlib import Path
import pickle

In [2]:
def effective_rank(A, eps=1e-12):
    """
    Compute the effective rank of a matrix A.

    Parameters
    ----------
    A : np.ndarray
        Input matrix.
    eps : float
        Small constant to avoid log(0).

    Returns
    -------
    erank : float
        Effective rank of A.
    singular_values : np.ndarray
        Singular values of A.
    """

    # Singular values
    singular_values = np.linalg.svd(A, compute_uv=False)

    # Normalize singular values into a probability distribution
    p = singular_values / (np.sum(singular_values) + eps)

    # Shannon entropy
    entropy = -np.sum(p * np.log(p + eps))

    # Effective rank
    erank = np.exp(entropy)

    return erank, singular_values

In [3]:
def cumulative_spectral_energy(A):
    """
    Compute cumulative spectral energy E_k^(2) of a matrix A.

    Parameters
    ----------
    A : np.ndarray
        Input matrix.

    Returns
    -------
    E : np.ndarray
        Cumulative spectral energy:
        E[k-1] = sum_{i=1}^k sigma_i^2 / sum_{i=1}^d sigma_i^2
    singular_values : np.ndarray
        Singular values of A.
    """

    # Singular values
    singular_values = np.linalg.svd(A, compute_uv=False)

    # Spectral energy
    energy = singular_values**2

    # Cumulative explained energy
    E = np.cumsum(energy) / np.sum(energy)

    return E, singular_values

In [4]:
# N_LAYERS = [3, 5]
N_LAYERS = [3, 5, 7, 9, 11, 13, 15, 17, 20]
ROLL_LOC = Path("Results/rollout_local")
ROLL_GLOB = Path("Results/rollout_global")

attn_rank_dict = {}
roll_rank_dict = {}
roll_energy_dict = {}

for layer in N_LAYERS:

    attn_ranks = []
    
    with open(ROLL_LOC / f"{layer}layers/attn_matrices{layer}layers.pkl", "rb") as f:
        attn_matrices = pickle.load(f)
    
    attn_list = [np.mean(attn_matrices[key]['attn_matrix'], axis=0) 
                 for key in attn_matrices.keys()]

    for matrix in attn_list:
        erank, _ = effective_rank(matrix)

        attn_ranks.append(erank)

    attn_rank_dict[f"{layer}layers"] = attn_ranks     

    rollouts = np.load(ROLL_GLOB / f"{layer}layers/rollout{layer}layers.npz")
    R = rollouts['rollout']
    erank_R, _ = effective_rank(R)
    energy_R, _ = cumulative_spectral_energy(R)
    roll_rank_dict[f"{layer}layers"] = erank_R
    roll_energy_dict[f"{layer}layers"] = energy_R

    # print(f"Layer {layer} --> Rank: {erank_R} - Energy: {energy_R.shape}")

# with open("Results/effective_ranks/attention_ranks.pkl", "wb") as f:
#     pickle.dump(attn_rank_dict, f)

# with open("Results/effective_ranks/rollout_ranks.pkl", "wb") as f:
#     pickle.dump(roll_rank_dict, f)

with open("Results/effective_ranks/rollout_energy.pkl", "wb") as f:
     pickle.dump(roll_energy_dict, f)